# Laboratorio AWS — Predicción de temperatura de un transformador

## Notebook 01 — Generación y exploración de datos

### Objetivo

Construir un conjunto de datos sintético que represente condiciones de operación de transformadores eléctricos y explorar visualmente la relación entre variables de operación, condiciones ambientales y temperatura del transformador.

**Este notebook no utiliza AWS, Amazon S3, boto3 ni SageMaker.** Su propósito es exclusivamente generar, validar y explorar la información que será utilizada posteriormente.

### Flujo del laboratorio

```text
Generación de datos
        ↓
Dataset sintético
        ↓
Validación básica
        ↓
Exploración estadística
        ↓
Visualización
        ↓
transformadores.csv
        ↓
Notebook 02 — Integración con Amazon S3
```

> Los datos son sintéticos y tienen fines exclusivamente pedagógicos.

## 1. Librerías

Se utilizan únicamente librerías locales para manipulación de datos, generación de números aleatorios y visualización.

No se requiere autenticación ni configuración de servicios de nube.

In [25]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

print("Librerías cargadas correctamente.")

## 2. Configuración de reproducibilidad

Se fija una semilla aleatoria para que el conjunto de datos sea reproducible.

In [26]:
np.random.seed(42)
N_OBSERVACIONES = 1000

print(f"Número de observaciones: {N_OBSERVACIONES}")
print("Semilla aleatoria: 42")

## 3. Generación de variables de operación y ambiente

Se generan 1.000 observaciones sintéticas.

| Variable | Descripción | Generación |
|---|---|---|
| `temperatura_ambiente` | Temperatura del entorno | Normal |
| `corriente` | Corriente eléctrica | Normal |
| `carga` | Nivel de carga | Uniforme |
| `humedad` | Humedad ambiental | Normal |
| `vibracion` | Nivel de vibración | Normal |
| `potencia` | Variable derivada | Calculada |
| `horas_operacion` | Horas acumuladas | Uniforme |

In [27]:
temperatura_ambiente = np.random.normal(25, 5, N_OBSERVACIONES)
corriente = np.random.normal(500, 80, N_OBSERVACIONES)
carga = np.random.uniform(50, 100, N_OBSERVACIONES)
humedad = np.random.normal(60, 10, N_OBSERVACIONES)
vibracion = np.random.normal(2.5, 0.5, N_OBSERVACIONES)
potencia = corriente * carga / 100
horas_operacion = np.random.uniform(100, 10000, N_OBSERVACIONES)

print("Variables generadas correctamente.")

## 4. Construcción de la variable objetivo

La variable objetivo es `temperatura_transformador`.

Para fines didácticos, se construye como una combinación de variables de operación y ambiente más un componente aleatorio que representa variaciones no explicadas.

Esto crea un problema de **regresión supervisada**:

```text
Variables predictoras
        ↓
temperatura_ambiente
corriente
carga
humedad
vibracion
potencia
horas_operacion
        ↓
Modelo de Machine Learning
        ↓
temperatura_transformador
```

In [28]:
temperatura_transformador = (
    35
    + 0.8 * temperatura_ambiente
    + 0.025 * corriente
    + 0.15 * carga
    + 0.08 * humedad
    + 1.5 * vibracion
    + 0.0005 * horas_operacion
    + np.random.normal(0, 2, N_OBSERVACIONES)
)

## 5. Construcción del DataFrame

Se consolidan las variables predictoras y la variable objetivo en un único DataFrame.

In [29]:
df = pd.DataFrame({
    "temperatura_ambiente": temperatura_ambiente,
    "corriente": corriente,
    "carga": carga,
    "humedad": humedad,
    "vibracion": vibracion,
    "potencia": potencia,
    "horas_operacion": horas_operacion,
    "temperatura_transformador": temperatura_transformador
})

print(f"Filas: {df.shape[0]}")
print(f"Columnas: {df.shape[1]}")
display(df.head())

## 6. Validación básica

Antes de utilizar los datos en Machine Learning se revisan dimensiones, tipos de datos, valores faltantes y estadísticas descriptivas.

In [30]:
print("Dimensiones:")
print(df.shape)

print("\nTipos de datos:")
display(df.dtypes.to_frame("tipo"))

print("\nValores faltantes:")
display(df.isna().sum().to_frame("valores_faltantes"))

print("\nEstadísticas descriptivas:")
display(df.describe().T)

## 7. Exploración visual: corriente y temperatura

La gráfica permite observar si existe una tendencia entre la corriente y la temperatura del transformador.

La visualización sirve para identificar patrones y dispersión; no demuestra causalidad.

In [31]:
plt.figure(figsize=(8, 5))
plt.scatter(
    df["corriente"],
    df["temperatura_transformador"],
    alpha=0.5
)
plt.xlabel("Corriente")
plt.ylabel("Temperatura del transformador")
plt.title("Relación entre corriente y temperatura")
plt.grid(alpha=0.2)
plt.show()

## 8. Exploración visual: carga y temperatura

Se analiza la relación entre el porcentaje de carga y la temperatura del transformador.

In [32]:
plt.figure(figsize=(8, 5))
plt.scatter(
    df["carga"],
    df["temperatura_transformador"],
    alpha=0.5
)
plt.xlabel("Carga (%)")
plt.ylabel("Temperatura del transformador")
plt.title("Relación entre carga y temperatura")
plt.grid(alpha=0.2)
plt.show()

## 9. Exploración visual: temperatura ambiente y temperatura del transformador

In [33]:
plt.figure(figsize=(8, 5))
plt.scatter(
    df["temperatura_ambiente"],
    df["temperatura_transformador"],
    alpha=0.5
)
plt.xlabel("Temperatura ambiente")
plt.ylabel("Temperatura del transformador")
plt.title("Relación entre temperatura ambiente y temperatura del transformador")
plt.grid(alpha=0.2)
plt.show()

## 10. Matriz de correlación

La correlación proporciona una primera aproximación a las relaciones lineales entre las variables.

No debe interpretarse como causalidad ni utilizarse como único criterio para seleccionar variables.

In [34]:
correlaciones = df.corr(numeric_only=True)
display(correlaciones.round(2))

plt.figure(figsize=(9, 7))
plt.imshow(correlaciones, aspect="auto")
plt.colorbar(label="Correlación")
plt.xticks(
    range(len(correlaciones.columns)),
    correlaciones.columns,
    rotation=45,
    ha="right"
)
plt.yticks(
    range(len(correlaciones.index)),
    correlaciones.index
)
plt.title("Matriz de correlación")
plt.tight_layout()
plt.show()

## 11. Guardar el dataset

El resultado de este notebook será:

```text
transformadores.csv
```

Este archivo será utilizado por el Notebook 02.

La separación de responsabilidades queda así:

- **Notebook 01:** generación y exploración.
- **Notebook 02:** Amazon S3 y modelado.

In [35]:
OUTPUT_FILE = "transformadores.csv"

df.to_csv(OUTPUT_FILE, index=False)

print(f"Dataset guardado correctamente: {OUTPUT_FILE}")
print(f"Filas: {len(df)}")
print(f"Columnas: {len(df.columns)}")

## 12. Verificación final

Se comprueba que el archivo fue generado y que puede volver a leerse correctamente.

In [36]:
from pathlib import Path

archivo = Path(OUTPUT_FILE)

if not archivo.exists():
    raise FileNotFoundError(f"No se encontró el archivo generado: {archivo}")

df_verificacion = pd.read_csv(archivo)

print(f"Archivo encontrado: {archivo.resolve()}")
print(f"Filas leídas: {df_verificacion.shape[0]}")
print(f"Columnas leídas: {df_verificacion.shape[1]}")

display(df_verificacion.head())

# Resumen del Notebook 01

Se construyó el punto de partida del laboratorio:

```text
Datos sintéticos
      ↓
Validación
      ↓
Exploración
      ↓
Visualización
      ↓
transformadores.csv
```

### Resultado

Se dispone de un dataset con variables ambientales, variables de operación, variables derivadas y una variable objetivo de temperatura.

### Siguiente paso

El **Notebook 02 — Integración con Amazon S3 y modelado** utilizará este archivo para:

1. conectarse a Amazon S3 mediante el IAM Role de SageMaker Studio;
2. cargar el dataset a `data/raw/`;
3. leer la información nuevamente desde S3;
4. crear los conjuntos `train` y `test`;
5. almacenarlos en S3;
6. entrenar varios modelos de regresión;
7. comparar sus métricas;
8. guardar los mejores modelos en `model/`;
9. almacenar métricas y predicciones en `output/`.